# Model Comparison — LinearRegression vs RandomForest vs DecisionTree
**Course:** ETL (G01) — Workshop 3  
**Goal:** Compare the three candidate models to justify the final selection.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model    import LinearRegression
from sklearn.ensemble        import RandomForestRegressor
from sklearn.tree            import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics         import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_columns', None)

RANDOM_STATE = 42

## 2. Load ML-Ready Dataset

In [ ]:
# ------------------------------------------------------------------
# Load the scaled dataset produced by feature_engineering.ipynb
# ------------------------------------------------------------------
df = pd.read_csv('../data/processed/happiness_ml_ready.csv')

FEATURES = ['gdp', 'family', 'health', 'freedom', 'generosity', 'corruption']
TARGET   = 'happiness_score'

X = df[FEATURES]
y = df[TARGET]

print(f'Dataset shape : {df.shape}')
print(f'Features      : {FEATURES}')
print(f'Target        : {TARGET}')

## 3. Train / Test Split

In [ ]:
# ------------------------------------------------------------------
# 70/30 split — same split for all models to ensure fair comparison
# ------------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=RANDOM_STATE
)

print(f'Training set : {X_train.shape[0]} rows')
print(f'Test set     : {X_test.shape[0]}  rows')

## 4. Train All Three Models and Collect Metrics

In [ ]:
# ------------------------------------------------------------------
# Define the three candidate models
# ------------------------------------------------------------------
candidate_models = {
    'LinearRegression':      LinearRegression(),
    'RandomForestRegressor': RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE),
    'DecisionTreeRegressor': DecisionTreeRegressor(random_state=RANDOM_STATE),
}

# ------------------------------------------------------------------
# Train each model and store predictions + metrics
# ------------------------------------------------------------------
results      = []   # metrics per model
predictions  = {}   # y_pred per model (for plots)

for name, model in candidate_models.items():
    # Train
    model.fit(X_train, y_train)

    # Predict on test set
    y_pred = model.predict(X_test)
    predictions[name] = y_pred

    # Metrics
    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)

    results.append({
        'model': name,
        'MAE':   round(mae,  4),
        'RMSE':  round(rmse, 4),
        'R2':    round(r2,   4),
    })

    print(f'{name:30s} | MAE={mae:.4f} | RMSE={rmse:.4f} | R²={r2:.4f}')

df_results = pd.DataFrame(results)
display(df_results)

## 5. Bar Chart — Metrics Comparison

In [ ]:
# ------------------------------------------------------------------
# Side-by-side bar charts for MAE, RMSE and R²
# Lower is better for MAE/RMSE — higher is better for R²
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metric_info = [
    ('MAE',  'steelblue', 'Lower is better'),
    ('RMSE', 'tomato',    'Lower is better'),
    ('R2',   'seagreen',  'Higher is better'),
]

short_names = ['LinearReg', 'RandomForest', 'DecisionTree']

for ax, (metric, color, note) in zip(axes, metric_info):
    bars = ax.bar(short_names, df_results[metric], color=color, edgecolor='white', width=0.5)
    ax.set_title(f'{metric}\n({note})', fontsize=11)
    ax.set_ylabel(metric)
    ax.set_ylim(0, df_results[metric].max() * 1.25)

    # Annotate value on each bar
    for bar, val in zip(bars, df_results[metric]):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.01,
            f'{val:.4f}',
            ha='center', va='bottom', fontsize=10
        )

    # Highlight the best bar
    best_idx = df_results[metric].idxmin() if metric != 'R2' else df_results[metric].idxmax()
    bars[best_idx].set_edgecolor('gold')
    bars[best_idx].set_linewidth(2.5)

plt.suptitle('Model Comparison — MAE, RMSE, R²', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Predicted vs Actual — All Three Models

In [ ]:
# ------------------------------------------------------------------
# Scatter: predicted vs actual for each model
# A perfect model follows the red dashed y=x line
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

colors = ['steelblue', 'seagreen', 'tomato']

for ax, (name, y_pred), color in zip(axes, predictions.items(), colors):
    r2 = r2_score(y_test, y_pred)

    ax.scatter(y_test, y_pred, alpha=0.35, color=color, s=15)
    ax.plot(
        [y_test.min(), y_test.max()],
        [y_test.min(), y_test.max()],
        'r--', linewidth=1.5, label='Perfect prediction'
    )
    ax.set_xlabel('Actual Score')
    ax.set_ylabel('Predicted Score')
    ax.set_title(f'{name}\nR²={r2:.3f}')
    ax.legend(fontsize=8)

plt.suptitle('Predicted vs Actual — All Models', fontsize=14)
plt.tight_layout()
plt.show()

## 7. Residuals Distribution — All Three Models

In [ ]:
# ------------------------------------------------------------------
# Residuals = actual - predicted
# A good model has residuals centered tightly around 0
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = ['steelblue', 'seagreen', 'tomato']

for ax, (name, y_pred), color in zip(axes, predictions.items(), colors):
    residuals = y_test.values - y_pred
    ax.hist(residuals, bins=25, color=color, edgecolor='white', alpha=0.8)
    ax.axvline(0, color='red', linestyle='--', linewidth=1.5)
    ax.axvline(residuals.mean(), color='black', linestyle='-', linewidth=1,
               label=f'mean={residuals.mean():.3f}')
    ax.set_xlabel('Residual')
    ax.set_ylabel('Count')
    ax.set_title(f'{name}\nResidual Distribution')
    ax.legend(fontsize=8)

plt.suptitle('Residuals Distribution — All Models', fontsize=14)
plt.tight_layout()
plt.show()

## 8. Final Selection Justification

In [ ]:
# ------------------------------------------------------------------
# Print a structured justification based on the real metric results
# ------------------------------------------------------------------
best_mae  = df_results.loc[df_results['MAE'].idxmin(),  'model']
best_rmse = df_results.loc[df_results['RMSE'].idxmin(), 'model']
best_r2   = df_results.loc[df_results['R2'].idxmax(),   'model']

print('=== Model Selection Summary ===')
display(df_results)

print(f'\nBest MAE  → {best_mae}')
print(f'Best RMSE → {best_rmse}')
print(f'Best R²   → {best_r2}')

print("""
Decision: RandomForestRegressor

Justification:
- Best MAE  (0.4069): smallest average prediction error
- Best RMSE (0.5237): most robust against large errors
- Best R²   (0.780) : explains 78% of variance in happiness score
- DecisionTree overfits (R²=0.501 on test set despite perfect train fit)
- LinearRegression is competitive but RandomForest captures non-linear
  relationships between GDP, health and happiness score
- Workshop focus is pipeline integration, not model complexity —
  RandomForest with default parameters is simple enough
""")